# 03 — Pipeline-depth RSS

Memory samples from passed leaves are summarized per run, then by depth. N is independent runs; units are MiB; this is descriptive diagnostic evidence. Missing depths remain PENDING, never zero. PMIC power is not used in this RSS view.


In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, passed_artifacts, pending_record, percentile_rows
from wafer_analysis.paths import resolve_analysis_batch

def resolve(experiment, env_name):
    return resolve_analysis_batch(experiment, os.environ.get(env_name))[0]

import csv
batch=resolve('e-perf-6','E_PERF_6_DIR'); records=[]
if batch is not None:
    for path in sorted(batch.rglob('memory.csv')):
        status=path.parent/'canonical-status.json'
        if not status.is_file(): continue
        import json
        if json.loads(status.read_text()).get('status')!='passed': continue
        with path.open() as stream:
            raw=list(csv.DictReader(stream))
        samples=[int(row['rss_bytes']) for row in raw if row.get('rss_bytes')] or [int(row['rss_kb'])*1024 for row in raw if row.get('rss_kb')]
        if samples: records.append({'condition':path.relative_to(batch).parts[0],'run':path.parent.name,'rss_mib':pd.Series(samples).median()/1048576})
df=pd.DataFrame(records); rows=[]
for depth in ['depth-1','depth-3','depth-5','depth-10']:
    values=df[df.condition==depth] if not df.empty else pd.DataFrame()
    if values.empty: rows.append(pending_record(depth,'no passed memory leaf','MiB RSS'))
    else: rows.append({'condition':depth,'status':'READY','N':len(values),'median_rss_mib':values.rss_mib.median(),'units':'MiB RSS','uncertainty':'descriptive only','thesis_evidence':False})
out=pd.DataFrame(rows); print('Power label where applicable: Raspberry Pi 5 PMIC internal-rail proxy; not total board power.'); display(out)
ready=out[out.status=='READY']
if not ready.empty:
    ax=ready.plot(x='condition',y='median_rss_mib',marker='o',legend=False); ax.set_ylabel('Median steady RSS (MiB)'); ax.set_title('Pipeline-depth RSS — diagnostic')
